# YZM5110 — Three baseline models

Run top to bottom on Colab. This notebook uses the [authors' published train and validation manifests](https://github.com/mmu-dermatology-research/isic-dicm-17k) and their `class` labels. The archive CSV's `diagnosis_1` is **not** used as a label or clinical feature.

**Interpretation limit:** The published train/validation manifests share 433 nonmissing patient IDs and 488 lesion IDs. Results here are **paper-split, reduced-backbone baselines**, not patient-independent clinical estimates or an exact reproduction of Ahammed et al.'s published score. The separate ISIC 2020 Kaggle test is not evaluated. See [data audit](https://github.com/i-Emin/Multimodal-Skin-Cancer-Detection/blob/main/reports/data_audit.md).

The fixed final epoch is reported. Outputs, including a SHA-256 digest of the embedded training source, settings, hardware, learning history, metrics, and predictions, are saved to your Google Drive. They are not automatically published to GitHub. The three models use the same seed, split, epoch count, batch size, and learning rate. Image-only and fusion use ImageNet-pretrained ResNet18; this smaller backbone and augmentation differ from the selected paper.


In [2]:
from pathlib import Path
import torch

print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running image models.')


Torch: 2.14.1+cu130
GPU: NVIDIA L4


## Dataset

If you already ran the audit notebook in the **same runtime**, this cell reuses `/content/isic-dicm-17k`. Otherwise it downloads and extracts the official 6.9 GB bundle. Colab's `/content` is temporary. Check the package terms on the [ISIC collection page](https://api.isic-archive.com/doi/isic-dicm-17k/).


In [3]:
import urllib.request, zipfile, shutil

data_dir = Path('/content/isic-dicm-17k')
image_dir = data_dir / 'images'
if not image_dir.is_dir() or not (data_dir / 'metadata.csv').is_file():
    archive = Path('/content/isic-dicm-17k.zip')
    if not archive.exists():
        print('Downloading official bundle (~6.9 GB)...', flush=True)
        urllib.request.urlretrieve(
            'https://isic-archive.s3.amazonaws.com/dois/10.34970-233480/isic-dicm-17k.zip', archive)
    print('Free disk GB:', round(shutil.disk_usage('/content').free / 1e9, 1))
    with zipfile.ZipFile(archive) as z:
        bad = [name for name in z.namelist() if Path(name).is_absolute() or '..' in Path(name).parts]
        if bad:
            raise ValueError(f'Unsafe archive paths: {bad[:3]}')
        z.extractall(data_dir)

images = list(image_dir.glob('*.jpg'))
print('JPG images:', len(images))
if len(images) != 17060:
    raise RuntimeError('Expected 17,060 JPG images. Check extraction before training.')


Free disk GB: 245.6
JPG images: 17060


## Save results to Drive

Mount your own Drive. This saves result files and final model checkpoints under `MyDrive/YZM5110/results/baselines/`. The raw dataset is not copied there. Checkpoints stay in your Drive and must not be committed to the public repository. If Google asks for Drive access, review the prompt before accepting.


In [4]:
from google.colab import drive
drive.mount('/content/drive')
out_dir = Path('/content/drive/MyDrive/YZM5110/results/baselines')
out_dir.mkdir(parents=True, exist_ok=True)
print('Output:', out_dir)


Mounted at /content/drive
Output: /content/drive/MyDrive/YZM5110/results/baselines


## Embedded versioned training code

The cell below is generated from `src/baseline.py`; do not edit it for this run. Its SHA-256 digest is saved in each `run.json` so results can be traced to exact code. The script version in GitHub is intended to match this cell once the repository sync completes.


In [5]:
"""Colab baseline runner for the authors' ISIC-DICM-17K train/valid manifests.

This uses a ResNet18 backbone and a short training budget. It is a documented
deviation from Ahammed et al. (2025), not an exact score reproduction.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             roc_auc_score)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18


BASE = "https://raw.githubusercontent.com/mmu-dermatology-research/isic-dicm-17k/main/"
MODES = ("image", "metadata", "fusion")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def load_manifests(image_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    train = pd.read_csv(BASE + "train-set-metadata.csv", low_memory=False)
    valid = pd.read_csv(BASE + "valid-set-metadata.csv", low_memory=False)
    needed = {"isic_id", "class", "age_approx", "sex", "anatom_site_general",
              "patient_id", "lesion_id"}
    for name, frame in (("train", train), ("valid", valid)):
        missing = needed - set(frame.columns)
        if missing:
            raise ValueError(f"{name} manifest missing columns: {sorted(missing)}")
        if frame["isic_id"].duplicated().any() or frame["isic_id"].isna().any():
            raise ValueError(f"{name} has duplicate or missing image IDs")
        if set(frame["class"].dropna().unique()) != {0, 1} or frame["class"].isna().any():
            raise ValueError(f"{name} requires complete binary class labels")
        missing_images = [x for x in frame["isic_id"] if not (image_dir / f"{x}.jpg").is_file()]
        if missing_images:
            raise FileNotFoundError(f"{name}: {len(missing_images)} images missing; first: {missing_images[:3]}")
    overlap = {}
    for col in ("isic_id", "patient_id", "lesion_id"):
        a = set(train[col].dropna().astype(str))
        b = set(valid[col].dropna().astype(str))
        overlap[col] = len(a & b)
    if overlap["isic_id"]:
        raise ValueError("Image IDs overlap across train and validation")
    audit = {
        "train_rows": len(train), "valid_rows": len(valid),
        "train_classes": train["class"].value_counts().sort_index().to_dict(),
        "valid_classes": valid["class"].value_counts().sort_index().to_dict(),
        "missing_patient_id": {"train": int(train.patient_id.isna().sum()),
                               "valid": int(valid.patient_id.isna().sum())},
        "distinct_cross_split_id_overlap": overlap,
        "warning": "Published validation is not patient/lesion independent when group overlap is nonzero.",
    }
    return train, valid, audit


class MetadataEncoder:
    """Fit age imputation and categorical vocabulary on training rows only."""

    def fit(self, train: pd.DataFrame) -> "MetadataEncoder":
        self.age_median = float(pd.to_numeric(train.age_approx, errors="coerce").median())
        self.age_mean = float(pd.to_numeric(train.age_approx, errors="coerce").fillna(self.age_median).mean())
        self.age_std = float(pd.to_numeric(train.age_approx, errors="coerce").fillna(self.age_median).std())
        if not np.isfinite(self.age_std) or self.age_std == 0:
            self.age_std = 1.0
        self.categories = {}
        for col in ("sex", "anatom_site_general"):
            values = train[col].fillna("unknown").astype(str).str.lower().str.strip()
            self.categories[col] = sorted(set(values) | {"unknown"})
        return self

    @property
    def dimension(self) -> int:
        return 1 + sum(len(v) for v in self.categories.values())

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        age = pd.to_numeric(frame.age_approx, errors="coerce").fillna(self.age_median)
        columns = [((age.to_numpy(dtype=np.float32) - self.age_mean) / self.age_std)[:, None]]
        for col, categories in self.categories.items():
            values = frame[col].fillna("unknown").astype(str).str.lower().str.strip().to_numpy()
            columns.append(np.column_stack([(values == category).astype(np.float32)
                                            for category in categories]))
        return np.concatenate(columns, axis=1).astype(np.float32)

    def config(self) -> dict:
        return {"age_median": self.age_median, "age_mean": self.age_mean,
                "age_std": self.age_std, "categories": self.categories,
                "dimension": self.dimension}


def image_transform(training: bool):
    # Paper: shortest side 224, center crop 224. Train augmentation below is
    # a practical ResNet18 deviation and is explicitly logged in run.json.
    if training:
        ops = [transforms.Resize(256),
                transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip()]
    else:
        ops = [transforms.Resize(224), transforms.CenterCrop(224)]
    ops += [transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])]
    return transforms.Compose(ops)


class LesionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, metadata: np.ndarray, image_dir: Path,
                 mode: str, training: bool):
        self.ids = frame.isic_id.astype(str).tolist()
        self.labels = frame["class"].astype(int).to_numpy()
        self.metadata = torch.from_numpy(metadata)
        self.image_dir = image_dir
        self.mode = mode
        self.transform = image_transform(training)

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, index: int):
        image = torch.empty(0)
        if self.mode != "metadata":
            with Image.open(self.image_dir / f"{self.ids[index]}.jpg") as source:
                image = self.transform(source.convert("RGB"))
        return image, self.metadata[index], int(self.labels[index]), self.ids[index]


class Baseline(nn.Module):
    def __init__(self, mode: str, metadata_dim: int, pretrained: bool = True):
        super().__init__()
        self.mode = mode
        if mode != "metadata":
            weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = resnet18(weights=weights)
            self.backbone.fc = nn.Identity()
        if mode != "image":
            self.meta_branch = nn.Sequential(nn.Linear(metadata_dim, 64), nn.ReLU(), nn.Dropout(0.2))
        width = 512 if mode == "image" else 64 if mode == "metadata" else 576
        self.head = nn.Sequential(nn.Linear(width, 128), nn.ReLU(), nn.Dropout(0.2),
                                  nn.Linear(128, 2))

    def forward(self, image: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:
        if self.mode == "image":
            features = self.backbone(image)
        elif self.mode == "metadata":
            features = self.meta_branch(meta)
        else:
            features = torch.cat([self.backbone(image), self.meta_branch(meta)], dim=1)
        return self.head(features)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    labels, probs, ids = [], [], []
    for image, meta, target, batch_ids in loader:
        image, meta = image.to(device), meta.to(device)
        with torch.autocast(device_type="cuda", enabled=device.type == "cuda"):
            logits = model(image, meta)
        probs.extend(torch.softmax(logits.float(), 1)[:, 1].cpu().tolist())
        labels.extend(target.tolist())
        ids.extend(batch_ids)
    y = np.asarray(labels, dtype=int)
    p = np.asarray(probs, dtype=float)
    pred = (p >= 0.5).astype(int)
    metrics = {
        "n": len(y), "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None,
        "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
        "per_class": classification_report(y, pred, labels=[0, 1], output_dict=True,
                                            zero_division=0),
    }
    predictions = pd.DataFrame({"isic_id": ids, "y_true": y, "p_class_1": p,
                                "y_pred": pred})
    return metrics, predictions


def run(args: argparse.Namespace) -> None:
    seed_everything(args.seed)
    image_dir = Path(args.data_dir) / "images"
    train, valid, audit = load_manifests(image_dir)
    print("Split audit:", json.dumps(audit, indent=2))
    encoder = MetadataEncoder().fit(train)
    train_data = LesionDataset(train, encoder.transform(train), image_dir, args.mode, True)
    valid_data = LesionDataset(valid, encoder.transform(valid), image_dir, args.mode, False)
    generator = torch.Generator().manual_seed(args.seed)
    train_loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.workers, pin_memory=torch.cuda.is_available(),
                              generator=generator)
    valid_loader = DataLoader(valid_data, batch_size=args.batch_size, shuffle=False,
                              num_workers=args.workers, pin_memory=torch.cuda.is_available())
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Baseline(args.mode, encoder.dimension).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    loss_fn = nn.CrossEntropyLoss()
    output = Path(args.output_dir) / args.mode
    output.mkdir(parents=True, exist_ok=True)
    commit = None
    if "__file__" in globals():
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                text=True, check=False).stdout.strip() or None
    code_sha = globals().get("SOURCE_SHA256")
    if code_sha is None and "__file__" in globals():
        code_sha = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    config = {
        "protocol": "authors_train_valid_manifest_group_overlap_present",
        "paper_score_reproduction": False, "mode": args.mode, "seed": args.seed,
        "epochs": args.epochs, "batch_size": args.batch_size, "lr": args.lr,
        "backbone": None if args.mode == "metadata" else "ImageNet pretrained ResNet18",
        "optimizer": "AdamW(weight_decay=1e-4)", "loss": "CrossEntropyLoss",
        "reported_epoch_rule": "fixed final epoch; no best-validation-epoch selection",
        "image_preprocessing": "train: resize256/random_resized_crop224/hflip; valid: shortest-side resize224/center_crop224; ImageNet normalization",
        "metadata_features": ["age_approx", "sex", "anatom_site_general"],
        "metadata_preprocessing": encoder.config(), "split_audit": audit,
        "data_dir": str(args.data_dir), "git_commit": commit,
        "source_sha256": code_sha,
        "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                     "pandas": pd.__version__, "numpy": np.__version__,
                     "sklearn": sklearn.__version__},
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    }
    (output / "run.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
    history = []
    started = time.perf_counter()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for image, meta, target, _ in train_loader:
            image, meta, target = image.to(device), meta.to(device), target.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=device.type == "cuda"):
                logits = model(image, meta)
                loss = loss_fn(logits, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses += float(loss.item()) * len(target)
            seen += len(target)
        metrics, predictions = evaluate(model, valid_loader, device)
        row = {"epoch": epoch, "train_loss": losses / seen,
               "seconds": time.perf_counter() - epoch_start,
               **{k: metrics[k] for k in ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc")}}
        history.append(row)
        pd.DataFrame(history).to_csv(output / "history.csv", index=False)
        print(args.mode, row, flush=True)
        # The fixed final epoch is the reported result. Selecting the best
        # validation epoch and reporting on that same set would add bias.
        if epoch == args.epochs:
            metrics["reported_epoch"] = epoch
            metrics["total_train_seconds"] = time.perf_counter() - started
            metrics["peak_gpu_memory_mb"] = (torch.cuda.max_memory_allocated() / 1e6
                                              if device.type == "cuda" else None)
            (output / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
            predictions.to_csv(output / "validation_predictions.csv", index=False)
            if args.save_checkpoint:
                torch.save(model.state_dict(), output / "final_model.pt")
    print("Saved", output, "total seconds", round(time.perf_counter() - started, 1))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=MODES, required=True)
    parser.add_argument("--data-dir", default="/content/isic-dicm-17k")
    parser.add_argument("--output-dir", default="/content/drive/MyDrive/YZM5110/results/baselines")
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--save-checkpoint", action="store_true")
    args = parser.parse_args()
    if args.epochs < 1 or args.batch_size < 1:
        parser.error("epochs and batch-size must be positive")
    run(args)



SOURCE_SHA256 = '05b186ebf3c28d47307fb553ad44c6e1af5affb7d5e3513e7bdf96c943be0946'
print('Training source SHA-256:', SOURCE_SHA256)


Training source SHA-256: 05b186ebf3c28d47307fb553ad44c6e1af5affb7d5e3513e7bdf96c943be0946


## Run the three matched baselines

Metadata-only runs first as a quick pipeline check. Image-only and fusion take longer. The code prints one row per epoch. Stop and share the full error if any run fails. Do not report a numerical result until all output files are written.


In [ ]:
import argparse

EPOCHS = 8
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
SEED = 42

for mode in ('metadata', 'image', 'fusion'):
    args = argparse.Namespace(mode=mode, data_dir=str(data_dir), output_dir=str(out_dir),
                              epochs=EPOCHS, batch_size=BATCH_SIZE,
                              lr=LEARNING_RATE, seed=SEED, workers=2,
                              save_checkpoint=True)
    print('RUNNING:', mode, flush=True)
    run(args)


RUNNING: metadata
Split audit: {
  "train_rows": 15098,
  "valid_rows": 1962,
  "train_classes": {
    "0": 7549,
    "1": 7549
  },
  "valid_classes": {
    "0": 981,
    "1": 981
  },
  "missing_patient_id": {
    "train": 7447,
    "valid": 1221
  },
  "distinct_cross_split_id_overlap": {
    "isic_id": 0,
    "patient_id": 433,
    "lesion_id": 488
  },
  "warning": "Published validation is not patient/lesion independent when group overlap is nonzero."
}
metadata {'epoch': 1, 'train_loss': 0.6567162652649868, 'seconds': 2.201461770999913, 'accuracy': 0.6498470948012233, 'balanced_accuracy': 0.6498470948012233, 'macro_f1': 0.6493616866300826, 'roc_auc': 0.6995415441814457}
metadata {'epoch': 2, 'train_loss': 0.6382839068195997, 'seconds': 1.716580729000043, 'accuracy': 0.6529051987767585, 'balanced_accuracy': 0.6529051987767585, 'macro_f1': 0.6522525134564168, 'roc_auc': 0.7050529894706873}
metadata {'epoch': 3, 'train_loss': 0.6352627272084148, 'seconds': 1.7426734900000156, 'accur

100%|██████████| 44.7M/44.7M [00:00<00:00, 230MB/s]


image {'epoch': 1, 'train_loss': 0.5143585251662759, 'seconds': 723.073828692, 'accuracy': 0.7874617737003058, 'balanced_accuracy': 0.7874617737003058, 'macro_f1': 0.787437422106452, 'roc_auc': 0.8854068275834122}
image {'epoch': 2, 'train_loss': 0.4620865086405178, 'seconds': 750.106607915, 'accuracy': 0.7813455657492355, 'balanced_accuracy': 0.7813455657492354, 'macro_f1': 0.7798632308475717, 'roc_auc': 0.8930292270779884}
image {'epoch': 3, 'train_loss': 0.4224610875096285, 'seconds': 747.1354029039999, 'accuracy': 0.8190621814475025, 'balanced_accuracy': 0.8190621814475025, 'macro_f1': 0.8190564938302137, 'roc_auc': 0.9009508905701706}
image {'epoch': 4, 'train_loss': 0.38674311586398447, 'seconds': 700.7985653190003, 'accuracy': 0.8287461773700305, 'balanced_accuracy': 0.8287461773700305, 'macro_f1': 0.8286885015856978, 'roc_auc': 0.911327973598265}
image {'epoch': 5, 'train_loss': 0.3459060538966287, 'seconds': 700.0412804429998, 'accuracy': 0.8134556574923547, 'balanced_accuracy

In [7]:
import argparse

args = argparse.Namespace(
    mode='fusion', data_dir=str(data_dir), output_dir=str(out_dir),
    epochs=8, batch_size=32, lr=1e-4, seed=42,
    workers=2, save_checkpoint=True
)
run(args)

Split audit: {
  "train_rows": 15098,
  "valid_rows": 1962,
  "train_classes": {
    "0": 7549,
    "1": 7549
  },
  "valid_classes": {
    "0": 981,
    "1": 981
  },
  "missing_patient_id": {
    "train": 7447,
    "valid": 1221
  },
  "distinct_cross_split_id_overlap": {
    "isic_id": 0,
    "patient_id": 433,
    "lesion_id": 488
  },
  "warning": "Published validation is not patient/lesion independent when group overlap is nonzero."
}
fusion {'epoch': 1, 'train_loss': 0.5117586370706084, 'seconds': 710.2344620539998, 'accuracy': 0.799184505606524, 'balanced_accuracy': 0.7991845056065239, 'macro_f1': 0.7991435980315227, 'roc_auc': 0.8934142177415753}
fusion {'epoch': 2, 'train_loss': 0.45232091082738934, 'seconds': 706.1931881359997, 'accuracy': 0.8017329255861366, 'balanced_accuracy': 0.8017329255861366, 'macro_f1': 0.8008721032622861, 'roc_auc': 0.9022731594484814}
fusion {'epoch': 3, 'train_loss': 0.41545861943665996, 'seconds': 738.818292109001, 'accuracy': 0.8256880733944955,

## Inspect measured results

These are your measured validation values. The notebook does not contain fabricated scores or an automatic claim of improvement. Keep the `YZM5110/results/baselines` folder in Drive; share `metrics.json`, `history.csv`, and `run.json` for each model so we can review them together.


In [8]:
import json
import pandas as pd

rows = []
for mode in ('metadata', 'image', 'fusion'):
    result = json.loads((out_dir / mode / 'metrics.json').read_text())
    rows.append({'mode': mode, 'epoch': result['reported_epoch'],
                 'macro_f1': result['macro_f1'],
                 'balanced_accuracy': result['balanced_accuracy'],
                 'accuracy': result['accuracy'], 'roc_auc': result['roc_auc'],
                 'train_minutes': result['total_train_seconds'] / 60,
                 'peak_gpu_MB': result['peak_gpu_memory_mb']})
display(pd.DataFrame(rows))
print('Files saved under:', out_dir)


,mode,epoch,macro_f1,balanced_accuracy,accuracy,roc_auc,train_minutes,peak_gpu_MB
0,metadata,8,0.657232,0.658002,0.658002,0.714507,0.241616,17.241600
1,image,8,0.819011,0.819062,0.819062,0.908464,96.314310,602.015232
2,fusion,8,0.813278,0.813965,0.813965,0.913518,95.524240,808.603136


Files saved under: /content/drive/MyDrive/YZM5110/results/baselines
